# Processing - testing notebook

For trying out preprocessing steps and parameters **one at a time**, in order.

**How chaining works.** There is one running stack called `current` and a running
name `current_tag`. Each step reads `current` and when you accept a result
`current` gets updated so the next step builds on it. This means **the order you run
the step cells is the pipeline order**: run median subtraction then contrast
enhancement and contrast enhancement operates on the median subtracted result.

Each step has three kinds of cell:
- **compute** - reads `current`, produces a candidate result (does *not* change `current` yet)
- **preview** - shows the candidate against `current`, auto-contrasted
- **accept & export** - saves the result and sets `current` to it so later steps chain on

If you only want to *test* a step without committing to it run compute + preview
and skip accept. To restart the chain from the raw data, run the **Reset** cell.

**Multiple variables** Trying multiple values (the Multiple variables cells) always
computes from `current` (the accepted chain so far) and shows the results side by side
without saving anything or changing `current`. So you can try several gammas on a median
subtracted stack, compare them and `current` stays the median subtracted stack. Once you
pick a value set it in that step's compute cell and run accept to commit it.

## Setup

In [1]:
import os
import numpy as np
import nd2
import tifffile as tiff
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from skimage.exposure import match_histograms
from skimage import exposure

In [ ]:
# ── Config ──
INPUT_PATH = r"//Hive2004/ag_idip/ZeljkaBaca_Data/ACID/250729/A07.3/H7_DENV2_MOI1_40h.nd2"
OUTPUT_DIR = "../output"           # where test exports go
POSITION   = 33                      # which position to test on (0-indexed)
TEST_FRAME = 40                      # which frame to preview (0-indexed)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# experiment name = folder the ND2 sits in (used in export filenames)
EXPERIMENT = os.path.basename(os.path.dirname(INPUT_PATH))
print("Experiment:", EXPERIMENT)

Experiment: A07.3


In [ ]:
# ── Load ── (run once)
# Handles ND2 files with or without a time axis:
#   (T, P, Y, X) timelapse  -> one position gives a (T, Y, X) stack
#   (P, Y, X) single frame  -> one position gives one (Y, X) image
#   a channel axis (C), if present, is reduced by taking CHANNEL (default 0)
#
# Reads the file's own axis labels, so a different axis order is handled correctly.
CHANNEL = 4        # which channel to keep if the file has a C axis (e.g. 0 = DAPI)

print("Loading ND2...")
with nd2.ND2File(INPUT_PATH) as f:
    sizes = f.sizes                       # e.g. {'T': 81, 'P': 42, 'Y': 1024, 'X': 1024}
    print(f"  sizes: {sizes}")
    arr = f.asarray().astype(np.float32)

axes = list(sizes.keys())

# collapse a channel axis first, if there is one
if "C" in axes:
    ci = axes.index("C")
    arr = np.take(arr, CHANNEL, axis=ci)
    axes.pop(ci)
    print(f"  took channel {CHANNEL}")

HAS_TIME = "T" in axes

if HAS_TIME:
    order = [axes.index(a) for a in ("T", "P", "Y", "X") if a in axes]
    data = np.transpose(arr, order)
    T, P, Y, X = data.shape
    print(f"  timelapse, shape {data.shape}")
    stack = data[:, POSITION]             # (T, Y, X) — steps work on this
    _tf = min(TEST_FRAME, T - 1)          # clamp preview frame to what exists
    if _tf != TEST_FRAME:
        print(f"  note: TEST_FRAME {TEST_FRAME} > last frame {T-1}, using {_tf}")
    frame = stack[_tf]                     # one frame for quick previews
    print(f"  position {POSITION+1}, preview frame {_tf}")
else:
    order = [axes.index(a) for a in ("P", "Y", "X") if a in axes]
    data = np.transpose(arr, order)
    P, Y, X = data.shape
    print(f"  single timepoint, shape {data.shape}")
    frame = data[POSITION]                # (Y, X)
    stack = frame[np.newaxis]             # (1, Y, X) so preview/export still work
    print(f"  position {POSITION+1} (no time axis) — one frame")

print("  HAS_TIME =", HAS_TIME)

In [ ]:
# ── Reset the chain ── (run to start over from the raw loaded stack)
current     = stack.copy()      # the running result every step builds on
current_tag = f"pos{POSITION+1:02d}"
print(f"chain reset -> {current_tag}, shape {current.shape}")

In [ ]:
# ── Helper functions ──

def _autoscale(img, low=1, high=99):
    """Percentile contrast limits, like FIJI's auto, ignores outlier pixels."""
    lo, hi = np.percentile(img, (low, high))
    if hi <= lo:
        lo, hi = float(img.min()), float(img.max())
    return lo, hi

def preview(before, after, title=""):
    """Show one frame before vs after, each auto-contrasted independently."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))
    blo, bhi = _autoscale(before)
    alo, ahi = _autoscale(after)
    ax[0].imshow(before, cmap="gray", vmin=blo, vmax=bhi)
    ax[0].set_title("before"); ax[0].axis("off")
    ax[1].imshow(after, cmap="gray", vmin=alo, vmax=ahi)
    ax[1].set_title(title or "after"); ax[1].axis("off")
    plt.tight_layout(); plt.show()

def export(stack_out, tag):
    """Save a (T, Y, X) stack to OUTPUT_DIR with a descriptive name."""
    fname = f"{EXPERIMENT}_pos{POSITION+1:02d}_{tag}.tif"
    path = os.path.join(OUTPUT_DIR, fname)
    tiff.imwrite(path, stack_out.astype(np.float32), imagej=True, metadata={"axes": "TYX"})
    print(f"saved: {path}")

def preview_video(before, after, title="", fps=10, step=1, scale=0.5, max_frames=40):
    """Play before vs after as synchronized videos, side by side.

    To keep it fast to build and light in the notebook, the clip is downsampled:
      step       take every Nth frame (2 = half the frames)
      scale      resize factor (0.5 = half resolution)
      max_frames hard cap on frames shown (overrides step if needed)
    Set step=1, scale=1 for full quality (slower).
    """
    from matplotlib import animation
    from IPython.display import HTML, display

    # frame subsampling
    idx = list(range(0, before.shape[0], max(1, step)))
    if len(idx) > max_frames:
        idx = idx[:: -(-len(idx) // max_frames)]
    b = before[idx]
    a = after[idx]

    # spatial downscale (simple striding — faster, needs no dependencies)
    if scale != 1:
        s = max(1, int(round(1 / scale)))
        b = b[:, ::s, ::s]
        a = a[:, ::s, ::s]

    T = b.shape[0]
    blo, bhi = _autoscale(before)          # limits from full data, not the subsample
    alo, ahi = _autoscale(after)

    fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
    im0 = ax[0].imshow(b[0], cmap="gray", vmin=blo, vmax=bhi, animated=True)
    im1 = ax[1].imshow(a[0], cmap="gray", vmin=alo, vmax=ahi, animated=True)
    ax[0].set_title("before"); ax[0].axis("off")
    ax[1].set_title(title or "after"); ax[1].axis("off")
    txt = fig.suptitle(f"frame 1 / {T}")
    plt.tight_layout()

    def update(t):
        im0.set_data(b[t]); im1.set_data(a[t])
        txt.set_text(f"frame {t+1} / {T}")
        return im0, im1, txt

    anim = animation.FuncAnimation(fig, update, frames=T, interval=1000/fps, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))


def show_variants(variants, ncols=4, frame=None):
    """Show one frame from each variant in a grid, to compare them.

    variants: list of (label, stack) pairs.
    frame:    which frame to show (defaults to TEST_FRAME).
    Each panel is auto-contrasted on its own whole-stack percentiles.
    Nothing is saved, this is only for looking.
    """
    if frame is None:
        frame = TEST_FRAME
    n = len(variants)
    ncols = min(ncols, n)
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.4))
    axes = np.atleast_1d(axes).ravel()
    for ax, (label, stk) in zip(axes, variants):
        lo, hi = _autoscale(stk)
        ax.imshow(stk[min(frame, stk.shape[0]-1)], cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

def average_reference(frames):
    """Reference image whose intensity distribution is the AVERAGE of all frames.

    frames: list/array of 2D images, all the same shape.
    Matching to this is the same as matching to the average histogram of the
    frames — an unbiased target, rather than one hand-picked frame.
    """
    stacked = np.stack([f.ravel() for f in frames])   # (N, num_pixels)
    stacked.sort(axis=1)                               # sort each frame's pixels
    avg_sorted = stacked.mean(axis=0)                  # average of the sorted values
    return avg_sorted.reshape(frames[0].shape)


def average_reference_streaming(frame_iter, shape, n):
    """Average intensity distribution over many frames without holding them all
    in memory — accumulates sorted pixels one frame at a time.

    frame_iter: yields 2D frames, all the same shape
    shape:      (Y, X) of a frame
    n:          number of frames the iterator yields
    Matching to the result = matching to the average histogram of all frames.
    """
    acc = np.zeros(int(np.prod(shape)), dtype=np.float64)
    for f in frame_iter:
        acc += np.sort(f.ravel())
    return (acc / n).reshape(shape).astype(np.float32)

In [ ]:
# ── Build & save the equalization reference (run ONCE) ──
REF_DIR   = "equalization_references"
TIME_EVERY = 1            # 1 = every frame; 5 = every 5th frame (all positions either way)
os.makedirs(REF_DIR, exist_ok=True)

if not HAS_TIME:
    raise RuntimeError("Reference build expects a timelapse (T, P, Y, X).")

ref_sc     = f"sc_{BG_METHOD}{BG_SIGMA}"
timepoints = list(range(0, T, TIME_EVERY))
pairs      = [(t, p) for p in range(P) for t in timepoints]
print(f"averaging {len(pairs)} frames ({P} positions x {len(timepoints)} timepoints)...")

def _ref_frames():
    done = 0
    for t, p in pairs:
        f = data[t, p]
        f = shadow_correction(f[np.newaxis], BG_SIGMA, BG_METHOD)[0]
        done += 1
        if done % 200 == 0:
            print(f"  {done}/{len(pairs)}", flush=True)
        yield f

reference = average_reference_streaming(_ref_frames(), (Y, X), len(pairs))

# name records: experiment, all positions, N timepoints, processing stage
ref_name = f"ref_{EXPERIMENT}_avg{P}pos_x{len(timepoints)}t_{ref_sc}"
ref_path = os.path.join(REF_DIR, ref_name + ".tif")
tiff.imwrite(ref_path, reference.astype(np.float32))
print(f"\nsaved: {ref_path}")
print(f"  shape {reference.shape}, mean {reference.mean():.1f}")

**Two ways to preview:** `preview(...)` shows a single frame (fast), and
`preview_video(...)` plays the whole clip before-vs-after with a play/scrub bar.
Any step's preview cell can use either — swap `preview` for `preview_video` and
pass the full stacks (`current`, `result`) instead of single frames.

---
## Shadow correction

Change `BG_METHOD` / `BG_SIGMA`, run, preview, export. Each export name records
the method and sigma.

In [ ]:
# --- change these ---
BG_METHOD = "gaussian"    # "gaussian", "median", "basic"
BG_SIGMA  = 8

# --- compute (reads current) ---
def shadow_correction(stack, sigma, method):
    if method in ("gaussian", "median"):
        out = np.empty_like(stack)
        for t in range(stack.shape[0]):
            f = stack[t]
            bg = gaussian_filter(f, sigma=sigma) if method == "gaussian" else median_filter(f, size=sigma)
            out[t] = f - bg + bg.mean()
        return out
    elif method == "basic":
        from basicpy import BaSiC
        b = BaSiC(); b.fit(stack)
        return (stack / b.flatfield[np.newaxis]).astype(np.float32)
    raise ValueError(method)

result = shadow_correction(current, BG_SIGMA, BG_METHOD)
result_tag = "sc_basic" if BG_METHOD == "basic" else f"sc_{BG_METHOD}{BG_SIGMA}"
print("computed:", result_tag)

In [ ]:
# --- preview (current vs candidate) ---
preview(current[TEST_FRAME], result[TEST_FRAME], result_tag)

# play the whole clip instead (before vs after, synchronized):
#preview_video(current, result, result_tag, fps=10)

In [ ]:
# --- accept & export (updates current so later steps chain on) ---
current = result
current_tag = f"{current_tag}_{result_tag}"
export(current, current_tag)

### Multiple variables - shadow correction

Loop over several sigma values (and/or methods) and export each.

Shows each variation as a grid to compare, nothing is saved. Once you pick a value set it in the step's compute cell and accept.

In [ ]:
# try multiple sigma values against `current` — VIEW ONLY (nothing saved)
VARY_METHOD = "gaussian"       # "gaussian" or "median"
variants = [(f"{VARY_METHOD} {s}", shadow_correction(current, s, VARY_METHOD))
            for s in [4, 8, 16, 32]]
show_variants(variants)

---
## Equalization

Histogram-match to a reference frame.

In [ ]:
# --- change these ---
EQUALIZE_TO = "saved"      # "saved"   = load a reference file built once (recommended)
                           # "frame"   = match to one frame (EQUALIZE_REF_T)
                           # "average" = build an average from the current stack now

EQUALIZE_REF_T   = 40      # used when EQUALIZE_TO = "frame"
AVG_FRAMES_EVERY = 10      # used when EQUALIZE_TO = "average"

# used when EQUALIZE_TO = "saved" — the file written by the build-once cell
REF_PATH = r"equalization_references/ref_A07.3_avg41pos_x81t_sc_gaussian8.tif"

# --- compute (reads current) ---
if not HAS_TIME:
    raise RuntimeError("Equalization needs a time axis — this file has none.")

if EQUALIZE_TO == "saved":
    reference = tiff.imread(REF_PATH).astype(np.float32)
    ref_id = os.path.splitext(os.path.basename(REF_PATH))[0].replace("ref_", "")
    result_tag = f"eq({ref_id})"
    print(f"  loaded reference: {os.path.basename(REF_PATH)}")

elif EQUALIZE_TO == "frame":
    reference = current[min(EQUALIZE_REF_T, current.shape[0]-1)]
    result_tag = f"eq{EQUALIZE_REF_T}"

elif EQUALIZE_TO == "average":
    ref_frames = [current[t] for t in range(0, current.shape[0], AVG_FRAMES_EVERY)]
    reference = average_reference(ref_frames)
    result_tag = f"eqavg{len(ref_frames)}"
    print(f"  averaged reference from {len(ref_frames)} frames")

else:
    raise ValueError(f"Unknown EQUALIZE_TO: {EQUALIZE_TO}")

result = np.empty_like(current)
for t in range(current.shape[0]):
    result[t] = match_histograms(current[t], reference)
print("computed:", result_tag)

In [ ]:
# --- preview (current vs candidate) ---
preview(current[TEST_FRAME], result[TEST_FRAME], result_tag)

# play the whole clip instead (before vs after, synchronized):
# preview_video(current, result, result_tag, fps=10)

In [ ]:
# --- accept & export ---
current = result
current_tag = f"{current_tag}_{result_tag}"
export(current, current_tag)

### Multiple variables - equalization

Loop over several reference frames and shows each variation as a grid to compare, nothing is saved. Once you pick a value set it in the step's compute cell and accept.

In [ ]:
# try multiple reference frames against `current` — VIEW ONLY (nothing saved)
def _eq(ref_t):
    r = current[min(ref_t, current.shape[0]-1)]
    out = np.empty_like(current)
    for t in range(current.shape[0]):
        out[t] = match_histograms(current[t], r)
    return out

variants = [(f"eq t={ref_t}", _eq(ref_t)) for ref_t in [0, 20, 40, 60, 80]]
show_variants(variants)

---
## Median subtraction

Change `MEDIAN_OFFSET` (`"auto"` = subtract as much as possible). Swap the input
stack if you want to test on a corrected stack.

In [ ]:
# --- change these ---
MEDIAN_OFFSET = "auto"    # "auto" or a number

# --- compute (reads current) ---
median_img = np.median(current, axis=0)
offset_used = float(median_img.min()) - 1 if MEDIAN_OFFSET == "auto" else MEDIAN_OFFSET
result = current - (median_img - offset_used)
result_tag = f"med{int(offset_used)}"
print("computed:", result_tag, f"(offset {int(offset_used)})")

In [ ]:
# --- preview (current vs candidate) ---
preview(current[TEST_FRAME], result[TEST_FRAME], result_tag)

# play the whole clip instead (before vs after, synchronized):
# preview_video(current, result, result_tag, fps=10)

In [ ]:
# --- accept & export ---
current = result
current_tag = f"{current_tag}_{result_tag}"
export(current, current_tag)

### Multiple variables - median subtraction

Loop over several offsets.

Shows each variation as a grid to compare, nothing is saved. Once you pick a value set it in the step's compute cell and accept.

In [ ]:
# try multiple offsets against `current` — VIEW ONLY (nothing saved)
median_img = np.median(current, axis=0)
def _med(off):
    o = float(median_img.min()) - 1 if off == "auto" else off
    return current - (median_img - o), int(o)

variants = []
for off in ["auto", 1000, 1300, 1600]:
    out, o = _med(off)
    variants.append((f"offset {o}", out))
show_variants(variants)

---
## Contrast enhancement

Pick a method, change its parameter, run, preview,
export. To sweep (e.g. several gammas): change the number, re-run compute +
export — each value lands as its own file.

- **gamma** — one number, `GAMMA` (<1 brightens, >1 darkens)
- **clahe** — `CLAHE_CLIP` (higher = stronger), `CLAHE_KERNEL`
- **percentile** — `PCT_LOW`, `PCT_HIGH`
- **sigmoid** — `SIGMOID_CUTOFF`, `SIGMOID_GAIN`

In [ ]:
# --- change these ---
CONTRAST_METHOD = "gamma"     # "clahe", "gamma", "percentile", "sigmoid"

GAMMA          = 0.7
CLAHE_CLIP     = 0.01
CLAHE_KERNEL   = 128
PCT_LOW        = 1.0
PCT_HIGH       = 99.0
SIGMOID_CUTOFF = 0.5
SIGMOID_GAIN   = 10

# --- compute (reads current) ---
def enhance(stack_in, method):
    lo, hi = float(stack_in.min()), float(stack_in.max())
    if hi <= lo:
        return stack_in
    out = np.empty_like(stack_in, dtype=np.float32)
    for t in range(stack_in.shape[0]):
        norm = np.clip((stack_in[t] - lo) / (hi - lo), 0, 1)
        if method == "clahe":
            e = exposure.equalize_adapthist(norm, kernel_size=CLAHE_KERNEL, clip_limit=CLAHE_CLIP)
        elif method == "gamma":
            e = exposure.adjust_gamma(norm, gamma=GAMMA)
        elif method == "percentile":
            p_lo, p_hi = np.percentile(norm, (PCT_LOW, PCT_HIGH))
            e = exposure.rescale_intensity(norm, in_range=(p_lo, p_hi))
        elif method == "sigmoid":
            e = exposure.adjust_sigmoid(norm, cutoff=SIGMOID_CUTOFF, gain=SIGMOID_GAIN)
        else:
            raise ValueError(method)
        out[t] = e * (hi - lo) + lo
    return out

result = enhance(current, CONTRAST_METHOD)
if CONTRAST_METHOD == "clahe":
    result_tag = f"ce_clahe{CLAHE_CLIP}"
elif CONTRAST_METHOD == "gamma":
    result_tag = f"ce_gamma{GAMMA}"
elif CONTRAST_METHOD == "percentile":
    result_tag = f"ce_pct{PCT_LOW}-{PCT_HIGH}"
elif CONTRAST_METHOD == "sigmoid":
    result_tag = f"ce_sig{SIGMOID_CUTOFF}g{SIGMOID_GAIN}"
print("computed:", result_tag)

In [ ]:
# --- preview (current vs candidate) ---
preview(current[TEST_FRAME], result[TEST_FRAME], result_tag)

# play the whole clip instead (before vs after, synchronized):
# preview_video(current, result, result_tag, fps=10)

In [ ]:
# --- accept & export ---
current = result
current_tag = f"{current_tag}_{result_tag}"
export(current, current_tag)

### Multiple variables - contrast enhancement

Shows each variation as a grid to compare, nothing is saved. Once you pick a value set it in the step's compute cell and accept.

In [ ]:
# try multiple contrast settings against `current`, VIEW ONLY (nothing saved)
# Pick which method to vary and its value list.
VARY = "gamma"      # "gamma", "clahe", "percentile", "sigmoid"

lo, hi = float(current.min()), float(current.max())
def _apply(fn):
    out = np.empty_like(current, dtype=np.float32)
    for t in range(current.shape[0]):
        norm = np.clip((current[t] - lo) / (hi - lo), 0, 1)
        out[t] = fn(norm) * (hi - lo) + lo
    return out

variants = []
if VARY == "gamma":
    for g in [0.5, 0.7, 0.9, 1.2]:
        variants.append((f"gamma {g}", _apply(lambda n, g=g: exposure.adjust_gamma(n, gamma=g))))
elif VARY == "clahe":
    for clip in [0.005, 0.01, 0.02, 0.04]:
        variants.append((f"clahe {clip}",
            _apply(lambda n, c=clip: exposure.equalize_adapthist(n, kernel_size=128, clip_limit=c))))
elif VARY == "percentile":
    for a, b in [(1, 99), (2, 98), (5, 95)]:
        variants.append((f"pct {a}-{b}",
            _apply(lambda n, a=a, b=b: exposure.rescale_intensity(n, in_range=tuple(np.percentile(n, (a, b)))))))
elif VARY == "sigmoid":
    for gain in [5, 10, 15]:
        variants.append((f"sigmoid g{gain}",
            _apply(lambda n, g=gain: exposure.adjust_sigmoid(n, cutoff=0.5, gain=g))))

show_variants(variants)

---
# Compare timelapse vs fixed vs fixed-stained

Bring three sources onto the **same intensity scale** so they can be compared
pixel-to-pixel:

1. **timelapse** - its last frame (already processed: shadow correction + median subtraction)
2. **fixed.nd2** - same position, one frame, no dirt
3. **fixed_stained.nd2** - the last channel, one frame

The fixed images have no dirt and no time axis so they skip median subtraction.
Instead they get the **same shadow correction** as the timelapse then are
**histogram-matched to the timelapse's processed last frame** so all three end
up on one scale. The matched-to frame is the reference; shadow correction is
applied to the fixed images first so the match is like-for-like.

### Settings for the comparison

In [ ]:
# timelapse: use the `current` you built above (its last frame is the reference).
# Make sure you have run the chain on the timelapse up to (and including) the step
# you want the fixed images matched to, typically shadow + median.
REF_FRAME = current[-1]                      # timelapse processed last frame = reference
print("reference (timelapse last frame): shape", REF_FRAME.shape,
      "mean", round(float(REF_FRAME.mean()), 1))

# fixed (unstained), same position
FIXED_PATH     = r"\\Hive2004\ag_idip\ZeljkaBaca_Data\ACID\250729\A07.3\H7_DENV2_MOI1_40h_fixed.nd2"
FIXED_POSITION = POSITION                    # same position as the timelapse

# fixed stained: 5 channels, we want the LAST one
FIXED_STAINED_PATH     = r"\\Hive2004\ag_idip\ZeljkaBaca_Data\ACID\250729\A07.3\H7_DENV2_MOI1_40h_fixed_stained.nd2"
FIXED_STAINED_POSITION = POSITION
STAINED_CHANNEL        = -1                   # last channel

### Load the two fixed images

Each is one frame `(Y, X)` for the chosen position (and channel).

In [ ]:
def load_fixed_frame(path, position, channel=None):
    """Load a single (Y, X) frame for one position (and optional channel)."""
    with nd2.ND2File(path) as f:
        print(f"  {os.path.basename(path)} sizes: {f.sizes}")
        arr = f.asarray().astype(np.float32)
    axes = list(f.sizes.keys())

    # collapse channel if asked and present
    if channel is not None and "C" in axes:
        ci = axes.index("C")
        arr = np.take(arr, channel, axis=ci)
        axes.pop(ci)

    # take the position
    if "P" in axes:
        pi = axes.index("P")
        arr = np.take(arr, position, axis=pi)
        axes.pop(pi)

    # if a stray T axis remains (single timepoint), drop it
    if "T" in axes:
        ti = axes.index("T")
        arr = np.take(arr, 0, axis=ti)
        axes.pop(ti)

    return arr   # (Y, X)

fixed         = load_fixed_frame(FIXED_PATH, FIXED_POSITION)
fixed_stained = load_fixed_frame(FIXED_STAINED_PATH, FIXED_STAINED_POSITION, STAINED_CHANNEL)
print("fixed:", fixed.shape, "| fixed_stained:", fixed_stained.shape)

### Process the fixed images

Same shadow correction as the timelapse, then histogram-match to the timelapse
last frame. No median subtraction (no dirt, no time axis).

In [ ]:
# uses BG_METHOD / BG_SIGMA from the shadow-correction step above
def process_fixed(img, reference):
    # shadow correction on a single frame (wrap to (1, Y, X) and back)
    sc = shadow_correction(img[np.newaxis], BG_SIGMA, BG_METHOD)[0]
    # histogram-match to the timelapse reference frame
    return match_histograms(sc, reference)

fixed_proc         = process_fixed(fixed, REF_FRAME)
fixed_stained_proc = process_fixed(fixed_stained, REF_FRAME)

print("means after processing:")
print(f"  timelapse ref : {REF_FRAME.mean():.1f}")
print(f"  fixed         : {fixed_proc.mean():.1f}")
print(f"  fixed_stained : {fixed_stained_proc.mean():.1f}")

### Compare — the three side by side

All on the same scale now. Shown with shared contrast limits (from the reference)
so brightness is directly comparable, not auto-scaled per panel.

In [ ]:
lo, hi = _autoscale(REF_FRAME)     # one shared scale for all three

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (label, im) in zip(ax, [
        ("timelapse (last frame)", REF_FRAME),
        ("fixed", fixed_proc),
        ("fixed_stained (last ch)", fixed_stained_proc)]):
    a.imshow(im, cmap="gray", vmin=lo, vmax=hi)
    a.set_title(label); a.axis("off")
plt.tight_layout(); plt.show()

### Export the processed fixed images (optional)

In [ ]:
EXPORT_COMPARISON = True
if EXPORT_COMPARISON:
    tiff.imwrite(os.path.join(OUTPUT_DIR, f"{EXPERIMENT}_pos{POSITION+1:02d}_fixed_matched.tif"),
                 fixed_proc.astype(np.float32))
    tiff.imwrite(os.path.join(OUTPUT_DIR, f"{EXPERIMENT}_pos{POSITION+1:02d}_fixed_stained_matched.tif"),
                 fixed_stained_proc.astype(np.float32))
    print("saved matched fixed images")